<!-- paired-role-banner -->
> 🟧 **실습 코드 · 왼쪽 창** — `TODO`/빈 구현을 직접 완성하세요. 정답을 보기 전에 입력·출력 shape를 먼저 예측하세요.

# 18. End-to-End RAG — 검색, 근거 답변, 인용, 평가

외부 LLM/API 없이 실행되는 작은 RAG 시스템을 직접 만듭니다.
query rewrite, persisted index, retrieval, token budget, prompt,
extractive fallback, citation, grounding/faithfulness heuristic,
no-answer, retrieval/generation 분리 평가를 한 pipeline으로 묶습니다.

In [ ]:
# 이 컴퓨터를 감지해 안전한 시작값을 확인합니다. 각 실습은 더 작은 값으로도 동작합니다.
from llm_engineering_lab.hardware import HardwareProfile, recommend_training_profile

LAB_HARDWARE = HardwareProfile.detect()
LAB_PROFILE = recommend_training_profile(LAB_HARDWARE)
print("hardware:", LAB_HARDWARE)
print("recommended profile:", LAB_PROFILE)

## 두 창 학습법

왼쪽 `notebooks/exercises/18_...ipynb`에서 `TODO`를 채우고
오른쪽 `notebooks/solutions/18_...ipynb`로 확인하세요.
함수 하나를 채울 때마다 정상 query와 빈 query를 함께 넣어 보세요. 답변 문장보다 `retrieved ids`, score, context,
citation mapping, abstention reason을 먼저 관찰합니다.

## RAG 경계와 관측값

```text
offline: JSONL → validate → TF-IDF embedding/index → artifact 저장
online : raw query → rewrite → retrieve top-k → context budget → grounded prompt
                              → local extractive answer → citations → faithfulness checks
eval   : retrieval(hit/rank)                  generation(citation/support/no-answer)
```

실제 LLM을 연결해도 이 경계는 유지합니다. 이 노트북의 로컬 답변기는 재현 가능한 기준선이자 API 장애 시 fallback입니다.

In [ ]:
# 공통 준비 코드입니다. 그대로 실행하세요.
import json, math, re, tempfile
from dataclasses import asdict, dataclass, field
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer


def find_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "data" / "raw" / "knowledge_base.jsonl").exists():
            return candidate
    raise RuntimeError("코딩연습 폴더 또는 그 하위에서 실행하세요.")


ROOT = find_root()
KB_PATH = ROOT / "data" / "raw" / "knowledge_base.jsonl"
print("knowledge_base=", KB_PATH)
print(
    "execution=CPU (sparse TF-IDF/NumPy RAG pipeline; GPU transfer would add overhead)"
)

## 1) 문서 로드와 searchable text

index artifact에 넣을 metadata는 JSON 직렬화 가능한 값으로 제한합니다.
검색용 문자열에는 title/category/keywords/content를 포함하지만,
답변 근거에는 원문 content를 사용합니다. 검색용 확장 텍스트와 인용 가능한 원문을 구분하세요.

<!-- api-explanations:start -->
<!-- api-explanations:keys=json.loads -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>json.loads(s, ...)</code></summary>

#### `json.loads(s, ...)`

- **역할:** JSON 문자열을 Python 객체로 역직렬화합니다.
- **입력·반환:** 문자열·bytes를 받고 dict·list 등의 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 메모리에 있는 JSON line이나 응답을 구조화합니다.
- **주의점:** 입력이 한 문서인지 JSON Lines의 한 줄인지 구분해야 합니다.
- **공식 문서:** [json.loads](https://docs.python.org/3/library/json.html#json.loads)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
def load_documents(path: Path) -> list[dict]:
    """TODO: utf-8-sig JSONL을 읽고 id/title/category/content/keywords와 id 고유성을 검증하세요."""
    ...


def searchable_text(document: dict) -> str:
    # TODO: 검색할 필드를 공백으로 결합하세요.
    ...


documents = load_documents(KB_PATH)

## 2) persisted sparse index

문자 n-gram TF-IDF를 교육용 embedding으로 사용합니다. `SearchHit`은
score와 rank뿐 아니라 source metadata를 함께 운반합니다.
index 저장 시 vectorizer/matrix와 documents를 분리해 쓰고, 로드 시 row 수 일치를 다시 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=dataclass,np.argsort,json.dumps -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 3개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>@dataclass(frozen=False, slots=False, ...)</code></summary>

#### `@dataclass(frozen=False, slots=False, ...)`

- **역할:** type annotation을 바탕으로 초기화·표현 메서드 등을 생성합니다.
- **입력·반환:** class decorator 설정을 받고 변환된 class를 반환합니다.
- **이 셀에서 쓰는 이유:** 실험 설정과 결과 record의 필드를 명시적으로 정의합니다.
- **주의점:** mutable 기본값은 `default_factory`를 쓰고 equality·frozen 의미를 확인합니다.
- **공식 문서:** [dataclass](https://docs.python.org/3/library/dataclasses.html#dataclasses.dataclass)

</details>

<details>
<summary><code>np.argsort(a, axis=-1, kind=None)</code></summary>

#### `np.argsort(a, axis=-1, kind=None)`

- **역할:** 정렬 결과를 만드는 index 순서를 반환합니다.
- **입력·반환:** 배열과 축·dtype 등의 설정을 받고 배열 또는 수치 결과를 반환합니다.
- **이 셀에서 쓰는 이유:** 데이터 shape과 수치 연산을 논문 실험에 맞게 구성·검증합니다.
- **주의점:** 축, dtype, 복사·view 여부와 빈 배열에서의 동작을 확인해야 합니다.
- **공식 문서:** [np.argsort](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html)

</details>

<details>
<summary><code>json.dumps(obj, ensure_ascii=True, indent=None, ...)</code></summary>

#### `json.dumps(obj, ensure_ascii=True, indent=None, ...)`

- **역할:** Python 객체를 JSON 문자열로 직렬화합니다.
- **입력·반환:** 객체를 받고 JSON 문자열을 반환합니다.
- **이 셀에서 쓰는 이유:** metric이나 sample을 파일·로그에 넣기 전 안정된 문자열로 만듭니다.
- **주의점:** 한글 보존에는 `ensure_ascii=False`가 필요하고 객체 종류가 제한됩니다.
- **공식 문서:** [json.dumps](https://docs.python.org/3/library/json.html#json.dumps)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
@dataclass(frozen=True)
class SearchHit:
    rank: int
    score: float
    document: dict


class PersistedTfidfIndex:
    def __init__(self, min_score: float = 0.0):
        self.min_score = min_score
        self.vectorizer = TfidfVectorizer(
            analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True
        )
        self.matrix = None
        self.documents = []

    def fit(self, records: list[dict]):
        """TODO: documents를 보존하고 searchable text를 fit_transform 하세요."""
        ...

    def search(self, query: str, top_k: int = 3) -> list[SearchHit]:
        """TODO: cosine top-k를 계산하고 min_score 이상만 SearchHit로 반환하세요."""
        ...

    def save(self, folder: Path) -> None:
        """TODO: model.joblib + documents.json을 저장하세요."""
        ...

    @classmethod
    def load(cls, folder: Path):
        """TODO: artifact를 읽고 row/document 정렬을 검증하세요."""
        ...

In [ ]:
# TODO: index를 fit하고 shape/vocabulary 크기를 출력하세요.
offline_index = ...
print(...)

## 3) offline artifact 저장 → online load

서비스 프로세스가 매번 fit하지 않도록 index를 저장합니다. 임시 폴더를 사용해 실습 후 자동 정리하며,
로드 전후 같은 query의 id와 score가 일치하는지 검사합니다. `joblib`은 신뢰할 수 있는 로컬 artifact만 로드하세요.

<!-- api-explanations:start -->
<!-- api-explanations:keys=Path,np.allclose -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>Path(*pathsegments)</code></summary>

#### `Path(*pathsegments)`

- **역할:** 운영체제에 맞는 파일 경로 객체를 만듭니다.
- **입력·반환:** 경로 조각을 받고 `Path` 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** Windows에서도 문자열 결합 없이 데이터·결과 경로를 구성합니다.
- **주의점:** 상대 경로는 현재 작업 디렉터리를 기준으로 해석됩니다.
- **공식 문서:** [Path](https://docs.python.org/3/library/pathlib.html#pathlib.Path)

</details>

<details>
<summary><code>np.allclose(a, b, rtol=1e-5, atol=1e-8, equal_nan=False)</code></summary>

#### `np.allclose(a, b, rtol=1e-5, atol=1e-8, equal_nan=False)`

- **역할:** 두 배열의 모든 원소가 허용 오차 안에서 가까운지 검사합니다.
- **입력·반환:** 배열 `a`, 기준 배열 `b`, 오차 설정을 받고 Python `bool` 하나를 반환합니다.
- **이 셀에서 쓰는 이유:** 부동소수점 계산 결과를 정확한 값이 아니라 수치적으로 같은지 확인합니다.
- **주의점:** 조건은 `abs(a-b) <= atol + rtol*abs(b)`입니다. `==`와 달리 오차를 허용하고, `b`가 기준이라 비대칭일 수 있으며 1보다 매우 작은 값에는 기본 `atol`이 너무 클 수 있습니다.

##### 관련 수식과 코드 연결

$$
\lvert a_i-b_i\rvert \le \mathrm{atol}+\mathrm{rtol}\,\lvert b_i\rvert
$$

- **기호:** $a_i$는 계산값, $b_i$는 기준값, `atol`은 절대 오차, `rtol`은 상대 오차 허용치입니다.
- **수식 → 코드:** `isclose`는 이 부등식을 원소마다 계산하고, `allclose`는 모든 결과에 논리 AND를 적용합니다.
- **직관:** 완전히 같은 비트인지가 아니라 실험 목적에 충분히 가까운지를 묻는 검사입니다.
- **shape 확인:** 두 입력은 broadcasting 가능해야 합니다. `isclose`는 broadcast shape, `allclose`는 scalar `bool`을 반환합니다.
- **공식 문서:** [np.allclose](https://numpy.org/doc/stable/reference/generated/numpy.allclose.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
INDEX_TEMP = tempfile.TemporaryDirectory()
INDEX_PATH = Path(INDEX_TEMP.name) / "support_index"
# TODO: offline_index를 저장하고 online_index로 다시 로드하세요.
...
assert INDEX_PATH.joinpath("model.joblib").exists()

## 4) query rewrite

rewrite는 원문을 버리지 않고 정규화된 query 뒤에 동의어 힌트를 추가합니다. 오류 코드처럼 중요한 식별자는 보존합니다.
규칙 기반 baseline은 동작이 투명하고, rewrite가 검색을 악화시킨 사례를 쉽게 회귀 테스트할 수 있습니다.

In [ ]:
REWRITE_RULES = {
    "로그인이 안": "로그인 비밀번호 계정 잠금",
    "돈이 두 번": "중복 결제 이중 결제",
    "택배가 안 와": "배송 조회 운송장 출고",
    "너무 많이 호출": "429 rate limit Retry-After",
    "키가 안 돼": "API 401 인증 키",
    "탈퇴": "계정 삭제 개인정보",
}


def rewrite_query(query: str) -> str:
    """TODO: whitespace 정리/구분자 중화 후 일치하는 동의어를 덧붙이세요."""
    ...

<!-- api-explanations:start -->
<!-- api-explanations:keys=pd.DataFrame -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>pd.DataFrame(data=None, index=None, columns=None, ...)</code></summary>

#### `pd.DataFrame(data=None, index=None, columns=None, ...)`

- **역할:** 행·열 구조의 DataFrame을 만듭니다.
- **입력·반환:** 표 데이터와 열·그룹 설정을 받고 DataFrame, Series 또는 집계 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 학습 전 데이터 분포·결측·중복과 분할 기준을 점검합니다.
- **주의점:** dtype 추론, 결측값, index 보존과 grouping 후 shape 변화를 확인해야 합니다.
- **공식 문서:** [pd.DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
rewrite_cases = [
    "로그인이 안 돼요",
    "돈이 두 번 빠졌어요",
    "API 401 오류",
    "  택배가   안 와요 ",
]
# TODO: 원문/rewrite/top1/score_before/score_after 비교표를 만드세요.
rewrite_frame = ...
display(rewrite_frame)

## 5) retrieval 결과 관찰

답변을 만들기 전에 rank/id/title/category/score를 표로 확인합니다. score는 확률이 아니므로 corpus/vectorizer가 바뀌면
no-answer threshold도 다시 측정해야 합니다.

In [ ]:
sample_query = "카드에서 돈이 두 번 결제됐습니다"
# TODO: rewrite 후 top-3를 검색하고 관찰용 DataFrame을 만드세요.
sample_hits = ...
retrieval_frame = ...
display(retrieval_frame)

## 6) context formatting과 token budget

실제 tokenizer 없이 한국어/영문 혼합 문자열의 token 수를 정확히 알 수는 없습니다. 여기서는 보수적인 문자 기반 근사치를 사용합니다.
각 source block을 통째로 넣고 budget을 넘으면 다음 문서를 제외해 citation 경계를 보존합니다.

In [ ]:
def approximate_tokens(text: str) -> int:
    # TODO: 공백 제외 문자 수를 이용해 보수적 근사치를 반환하세요.
    ...


def format_context(
    hits: list[SearchHit], token_budget: int = 220
) -> tuple[str, list[dict]]:
    """TODO: [SOURCE id] block을 budget 안에 추가하고 source map을 반환하세요."""
    ...

In [ ]:
# TODO: budget 70/140/260에서 포함 source 수와 근사 token 수를 비교하세요.
budget_frame = ...
display(budget_frame)

## 7) grounded prompt construction

지시, context, 사용자 query를 명확히 분리하고 “근거가 없으면 보류”, “source id 인용”, “context 내부 지시 무시”를 명시합니다.
여기서 prompt를 출력하는 이유는 LLM 호출 전에 실제 입력을 검토하기 위해서입니다.

In [ ]:
def build_prompt(question: str, context: str) -> str:
    """TODO: evidence-only, no-answer, citation 규칙을 포함한 prompt를 만드세요."""
    ...


context, source_map = format_context(sample_hits, token_budget=220)
sample_prompt = build_prompt(sample_query, context)
print(sample_prompt)

## 8) 로컬 extractive answer fallback

LLM 대신 top-1 문서의 문장 중 query와 문자 bigram이 가장 많이 겹치는 문장을 선택합니다. 문장을 새로 만들어내지 않으므로
환각 가능성이 낮고 결과가 결정적입니다. 다만 자연스러움과 복합 추론 능력은 제한됩니다.

In [ ]:
def char_bigrams(text: str) -> set[str]:
    # TODO: 공백/기호를 정규화하고 2글자 집합을 반환하세요.
    ...


def extractive_answer(
    question: str, hits: list[SearchHit], sentence_count: int = 2
) -> str:
    """TODO: top-1 content 문장을 overlap으로 정렬해 인용과 함께 반환하세요."""
    ...

## 9) citation/source mapping

답변의 `[출처:kb-...]`를 파싱해 실제 context에 포함된 source인지 확인합니다. 존재하지 않는 id 인용, 인용 누락,
검색은 되었지만 budget 때문에 context에서 빠진 문서 인용을 모두 실패로 취급합니다.

In [ ]:
def cited_ids(answer: str) -> list[str]:
    # TODO: 정규식으로 [출처:...] id를 추출하세요.
    ...


def citation_report(answer: str, source_map: list[dict]) -> dict:
    """TODO: cited/allowed/unknown/valid를 반환하세요."""
    ...

## 10) grounding/faithfulness heuristic

자동 지표는 진실을 보장하지 않습니다. 여기서는 답변(인용 제거)의 문자 bigram 중 context에도 있는 비율을 support ratio로 봅니다.
extractive baseline에는 유용하지만 동의어를 쓰는 정상 답변을 과소평가할 수 있으므로 샘플 수동 검토와 함께 사용합니다.

In [ ]:
def support_ratio(answer: str, context: str) -> float:
    """TODO: 인용을 제거한 answer bigram 중 context에 포함된 비율을 계산하세요."""
    ...


grounded_answer = extractive_answer(sample_query, sample_hits)
# TODO: grounded/unsupported 답변의 비율을 비교하세요.
...

## 11) RAG pipeline class

online 경로를 하나의 명확한 인터페이스로 묶습니다. 반환 객체에 답변뿐 아니라 original/rewritten query, hits, context,
prompt, sources, diagnostics를 남겨 디버깅과 평가가 가능하게 합니다.

In [ ]:
@dataclass
class RAGResponse:
    status: str
    answer: str
    original_query: str
    rewritten_query: str
    hits: list[SearchHit] = field(default_factory=list)
    context: str = ""
    prompt: str = ""
    sources: list[dict] = field(default_factory=list)
    diagnostics: dict = field(default_factory=dict)


class RAGPipeline:
    def __init__(
        self,
        index,
        top_k: int = 3,
        token_budget: int = 220,
        min_top_score: float = 0.12,
    ):
        # TODO: 설정을 보존하세요.
        ...

    def run(self, question: str) -> RAGResponse:
        """TODO: rewrite→retrieve→threshold→context→prompt→answer→검증을 연결하세요."""
        ...

In [ ]:
pipeline = RAGPipeline(online_index, top_k=3, token_budget=220, min_top_score=0.12)
# TODO: 중복 결제 질문을 실행하고 상태/답변/source/diagnostics를 출력·검증하세요.
response = ...

## 12) no-answer 처리

빈 질문, 도메인 밖 질문, 낮은 retrieval score, 너무 작은 context budget을 구분합니다. no-answer는 실패가 아니라
근거 없는 답변을 막는 제품 기능입니다. threshold는
answerable/unanswerable validation set의 coverage와 오답률로 결정하세요.

In [ ]:
no_answer_queries = ["", "양자 얽힘 실험 장비 예약", "오늘 서울 날씨와 주가를 알려줘"]
# TODO: 각 query의 status/reason/top_score를 표로 만드세요.
no_answer_frame = ...
display(no_answer_frame)

## 13) retrieval 평가와 generation 평가를 분리하기

정답 문서가 top-k에 없으면 retrieval 실패입니다. 정답 문서가 있는데
인용이 틀리거나 근거에 없는 문장을 만들면 generation/grounding
실패입니다. 아래 표는 `retrieval_hit`, `answered`, `citation_valid`,
`support_ratio`를 한 행에 두되 서로 다른 품질 축으로 유지합니다.

In [ ]:
EVAL_CASES = [
    {
        "query": "비밀번호 인증 링크가 안 와요",
        "expected_id": "kb-001",
        "answerable": True,
    },
    {
        "query": "계정이 5번 실패 후 잠겼어요",
        "expected_id": "kb-002",
        "answerable": True,
    },
    {"query": "돈이 두 번 결제됐어요", "expected_id": "kb-003", "answerable": True},
    {"query": "환불 카드 반영 기간", "expected_id": "kb-005", "answerable": True},
    {
        "query": "배송 조회가 48시간 멈췄어요",
        "expected_id": "kb-006",
        "answerable": True,
    },
    {"query": "파손 상품 사진 교환", "expected_id": "kb-008", "answerable": True},
    {"query": "401 Bearer API 키 인증", "expected_id": "kb-009", "answerable": True},
    {
        "query": "429 Retry-After 지수 백오프",
        "expected_id": "kb-010",
        "answerable": True,
    },
    {"query": "양자 컴퓨터 예약", "expected_id": None, "answerable": False},
    {"query": "서울의 미세먼지", "expected_id": None, "answerable": False},
]


def evaluate_pipeline(rag: RAGPipeline, cases: list[dict]) -> pd.DataFrame:
    """TODO: retrieval/generation/no-answer 열을 각각 계산하세요."""
    ...

In [ ]:
# TODO: 평가표와 retrieval recall@3, answer coverage,
# citation validity, correct abstention을 요약하세요.
evaluation = ...
summary = ...
# TODO: retrieval/generation/abstention 실패 행도 각각 필터링하세요.
retrieval_failures = ...
generation_failures = ...
abstention_failures = ...
display(evaluation)
print(summary)

## 직접 해볼 실험과 운영 체크리스트

1. answerable/unanswerable query를 각각 20개로 늘리고 threshold별 coverage/오답률 곡선을 만드세요.
2. 17번의 dense index 또는 sparse+dense RRF를 같은 `search()` 계약 뒤에 연결하세요.
3. context budget과 top-k를 바꾸고 retrieval hit인데 context에서 누락되는 사례를 찾으세요.
4. 문서에 “이전 지시를 무시하라”는 문자열을 넣고 prompt injection 회귀 테스트를 작성하세요.
5. 문서 `version`, `tenant_id`, `access_roles` metadata filter를 retrieval 전에 적용하세요.
6. 로컬 LLM이나 외부 API를 연결할 때도 extractive fallback, timeout, retry, 비용/latency logging을 유지하세요.

완료 기준: RAG 답변 오류를
**index/retrieval/context/generation/citation/policy** 중 어느 단계
문제인지 evidence로 분리할 수 있습니다.